In [ ]:
from pathlib import Path
PROJECT = Path(r'D:\tcc_ibov')
DIRS = {
    'raw':       PROJECT / 'data' / 'raw',
    'processed': PROJECT / 'data' / 'processed',
}

import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) TCC-research"}


In [2]:
import time

def fetch(url, timeout=120, tries=3):
    """Timeout longo p/ sitemaps gerados dinamicamente; mede o tempo
    de resposta p/ diagnosticarmos se é lentidão ou tarpit."""
    for i in range(1, tries + 1):
        t0 = time.time()
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            print(f"[{i}] {r.status_code} em {time.time()-t0:.1f}s | {len(r.content):,} bytes")
            if r.status_code == 200:
                return r
        except requests.exceptions.Timeout:
            print(f"[{i}] timeout após {time.time()-t0:.1f}s")
        except requests.exceptions.RequestException as e:
            print(f"[{i}] erro: {type(e).__name__}")
        time.sleep(5)
    return None

resp = fetch("https://www.infomoney.com.br/sitemap_index.xml")
if resp:
    print("\n--- Primeiros 3000 chars ---")
    print(resp.text[:3000])
    print("\n--- Últimos 1500 chars ---")
    print(resp.text[-1500:])

[1] 200 em 0.2s | 88,470 bytes

--- Primeiros 3000 chars ---
<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" href="//www.infomoney.com.br/wp-content/plugins/wordpress-seo/css/main-sitemap.xsl"?>
<sitemapindex xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
	<sitemap>
		<loc>https://www.infomoney.com.br/post-sitemap.xml</loc>
		<lastmod>2026-08-03T00:40:35+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://www.infomoney.com.br/post-sitemap2.xml</loc>
		<lastmod>2001-01-05T09:36:00+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://www.infomoney.com.br/post-sitemap3.xml</loc>
		<lastmod>2001-02-05T13:22:00+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://www.infomoney.com.br/post-sitemap4.xml</loc>
		<lastmod>2001-03-06T11:33:00+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://www.infomoney.com.br/post-sitemap5.xml</loc>
		<lastmod>2001-04-02T12:00:00+00:00</lastmod>
	</sitemap>
	<sitemap>
		<loc>https://www.infomoney.com.br/post-sitemap6.xml</

In [3]:
import xml.etree.ElementTree as ET
import pandas as pd
import re

NS = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
root = ET.fromstring(resp.content)

entries = []
for sm in root.findall('sm:sitemap', NS):
    loc = sm.find('sm:loc', NS).text
    lastmod = sm.find('sm:lastmod', NS)
    entries.append({'loc': loc, 'lastmod': lastmod.text if lastmod is not None else None})

idx = pd.DataFrame(entries)
idx['lastmod'] = pd.to_datetime(idx['lastmod'], utc=True, errors='coerce')


mask = idx['loc'].str.contains(r'/post-sitemap\d*\.xml$')
posts = idx[mask].copy()

posts['n'] = (posts['loc'].str.extract(r'post-sitemap(\d*)\.xml')[0]
              .replace('', '1').astype(int))
posts = posts.sort_values('n').reset_index(drop=True)

print(f"Total de post-sitemaps: {len(posts)}")
print(f"Range temporal: {posts['lastmod'].min()} -> {posts['lastmod'].max()}")

print("\nSitemaps por ano (via lastmod):")
print(posts['lastmod'].dt.year.value_counts().sort_index().tail(12))
lo = pd.Timestamp('2018-06-01', tz='UTC')
hi = pd.Timestamp('2025-02-01', tz='UTC')
sel = posts[(posts['lastmod'] >= lo) & (posts['lastmod'] <= hi)]

first_n = max(sel['n'].min() - 1, 1)
last_n  = sel['n'].max()
target = posts[(posts['n'] >= first_n) & (posts['n'] <= last_n)]

print(f"\nSitemaps a baixar: nº {first_n} a {last_n} ({len(target)} arquivos)")
print(f"Estimativa de URLs: ~{len(target) * 1000:,} (Yoast usa ~1000/arquivo)")

target.to_parquet(DIRS['raw'] / 'sitemap_targets.parquet')
print("Salvo: sitemap_targets.parquet")

Total de post-sitemaps: 571
Range temporal: 2001-01-05 09:36:00+00:00 -> 2026-08-03 00:40:35+00:00

Sitemaps por ano (via lastmod):
lastmod
2015    18
2016    16
2017    15
2018    13
2019    35
2020    12
2021    13
2022    18
2023    25
2024    32
2025    45
2026    32
Name: count, dtype: int64

Sitemaps a baixar: nº 353 a 498 (146 arquivos)
Estimativa de URLs: ~146,000 (Yoast usa ~1000/arquivo)
Salvo: sitemap_targets.parquet


In [4]:
f = DIRS['raw'] / 'infomoney_urls_raw.parquet'

if f.exists():
    urls = pd.read_parquet(f)
else:
    parts = sorted((DIRS['raw'] / 'sitemaps_out').glob('*.parquet'))
    print(f"Arquivos: {len(parts)} (esperado: 146)")
    urls = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    urls = urls.drop_duplicates(subset='url').dropna(subset=['url'])
    urls['lastmod'] = pd.to_datetime(urls['lastmod'], utc=True, errors='coerce')
    urls.to_parquet(f)
    print(f"Consolidado salvo: {f}")

urls['secao'] = urls['url'].str.extract(r'infomoney\.com\.br/([^/]+)/')

print(f"URLs únicas: {len(urls):,}")
print(f"lastmod nulos: {urls['lastmod'].isna().sum():,} "
      f"({urls['lastmod'].isna().mean():.1%})")

print("\nTop 15 seções:")
print(urls['secao'].value_counts().head(15))

print("\nURLs por ano (lastmod):")
print(urls['lastmod'].dt.year.value_counts().sort_index())

URLs únicas: 145,765
lastmod nulos: 0 (0.0%)

Top 15 seções:
secao
mercados           63903
politica           19043
economia           15501
onde-investir       9654
minhas-financas     7438
carreira            7387
consumo             6920
negocios            6350
mundo               4549
business            3762
stock-pickers        723
saude                156
brasil               145
advisor              145
esportes              87
Name: count, dtype: int64

URLs por ano (lastmod):
lastmod
2018     8358
2019    35011
2020    11716
2021    13836
2022    17824
2023    24844
2024    31494
2025     2682
Name: count, dtype: int64


In [5]:
SECOES_CORPUS = ['mercados', 'economia']      # decisão editorial — parametrizada
SECOES_RESERVA = ['politica']                  # guardadas p/ ablação futura

lo = pd.Timestamp('2018-06-01', tz='UTC')      # margem: lastmod é aproximado;
hi = pd.Timestamp('2025-02-01', tz='UTC')      # o corte fino vem do datePublished

fila = urls[
    urls['secao'].isin(SECOES_CORPUS)
    & urls['lastmod'].between(lo, hi)
].drop_duplicates(subset='url').reset_index(drop=True)

reserva = urls[
    urls['secao'].isin(SECOES_RESERVA)
    & urls['lastmod'].between(lo, hi)
].drop_duplicates(subset='url').reset_index(drop=True)

print(f"Fila principal: {len(fila):,} URLs")
print(fila['secao'].value_counts())
print(f"\nReserva (politica): {len(reserva):,} URLs")

fila.to_parquet(DIRS['raw'] / 'fila_artigos.parquet')
reserva.to_parquet(DIRS['raw'] / 'reserva_politica.parquet')
print("\nSalvos: fila_artigos.parquet, reserva_politica.parquet")

Fila principal: 78,685 URLs
secao
mercados    63184
economia    15501
Name: count, dtype: int64

Reserva (politica): 18,694 URLs

Salvos: fila_artigos.parquet, reserva_politica.parquet


In [6]:
"""Baixa 12 artigos espalhados pelo período e testa a extração de
title + datePublished do JSON-LD. Só liberamos a coleta em massa
se 12/12 extraírem timestamp com timezone."""

import json
import re
import time

import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) TCC-research"}

fila = pd.read_parquet('/content/drive/MyDrive/tcc_ibov/data/raw/fila_artigos.parquet')   # copie do Drive

# Amostra estratificada: 2 URLs por ano de lastmod, 2018-2024 —
# se a estrutura da página mudou ao longo dos anos, é assim que descobrimos.
fila['ano'] = fila['lastmod'].dt.year
amostra = (fila[fila['ano'].between(2018, 2024)]
           .groupby('ano', group_keys=False)
           .apply(lambda g: g.sample(2, random_state=42)))

def extrai_meta(html: str) -> dict:
    """title + datas via JSON-LD; fallback p/ meta tags OpenGraph."""
    out = {'headline': None, 'datePublished': None, 'dateModified': None, 'via': None}

    # 1) JSON-LD (fonte de verdade)
    for m in re.finditer(
            r'<script[^>]*type=["\']application/ld\+json["\'][^>]*>(.*?)</script>',
            html, re.DOTALL):
        try:
            data = json.loads(m.group(1))
        except json.JSONDecodeError:
            continue
        # JSON-LD pode ser objeto único, lista, ou @graph
        candidatos = data if isinstance(data, list) else \
                     data.get('@graph', [data]) if isinstance(data, dict) else []
        for c in candidatos:
            if isinstance(c, dict) and c.get('@type') in (
                    'NewsArticle', 'Article', 'ReportageNewsArticle'):
                out.update({
                    'headline': c.get('headline'),
                    'datePublished': c.get('datePublished'),
                    'dateModified': c.get('dateModified'),
                    'via': 'json-ld',
                })
                return out

    # 2) Fallback: meta tags
    mt = re.search(r'<meta[^>]+property=["\']article:published_time["\'][^>]+'
                   r'content=["\']([^"\']+)', html)
    tt = re.search(r'<meta[^>]+property=["\']og:title["\'][^>]+content=["\']([^"\']+)', html)
    if mt:
        out.update({'datePublished': mt.group(1),
                    'headline': tt.group(1) if tt else None,
                    'via': 'meta-og'})
    return out

resultados = []
for _, row in amostra.iterrows():
    try:
        r = requests.get(row['url'], headers=HEADERS, timeout=60)
        meta = extrai_meta(r.text)
        meta.update({'url': row['url'], 'status': r.status_code,
                     'bytes': len(r.content)})
    except requests.exceptions.RequestException as e:
        meta = {'url': row['url'], 'status': type(e).__name__}
    resultados.append(meta)
    print(f"[{meta.get('status')}] {meta.get('via')} | {meta.get('datePublished')} | "
          f"{str(meta.get('headline'))[:60]}")
    time.sleep(2)

df = pd.DataFrame(resultados)
ok = df['datePublished'].notna().sum()
print(f"\n{ok}/{len(df)} com datePublished extraído")
print(f"Timezones encontrados: "
      f"{df['datePublished'].str.extract(r'([+-]\d{2}:\d{2}|Z)$')[0].unique()}")

/tmp/ipykernel_2069/2810424787.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(2, random_state=42)))


[200] json-ld | 2018-11-05T09:13:00+00:00 | Ibovespa Futuro sobe com ajuste de ADRs e otimismo com nomea
[200] json-ld | 2018-06-28T12:07:00+00:00 | Ipea reduz de 3,0% para 1,7% expectativa de crescimento do P
[200] json-ld | 2019-02-22T09:39:00+00:00 | Ibovespa Futuro sobe de olho em reunião entre EUA e China e 
[200] json-ld | 2019-11-22T12:10:19+00:00 | Itaú confirma multa bilionária da prefeitura de SP, Braskem 
[200] json-ld | 2020-09-24T14:44:20+00:00 | Bradesco BBI reitera visão positiva para ações brasileiras, 
[200] json-ld | 2019-06-28T08:50:00+00:00 | Trump diz a Bolsonaro que visitará o Brasil
[200] json-ld | 2021-02-01T13:50:44+00:00 | Americanas e B2W fazem doação de R$ 5 milhões para fábrica d
[200] json-ld | 2021-09-16T19:15:48+00:00 | Foxbit lança sistema automatizado para operações com criptom
[200] json-ld | 2022-01-25T11:57:04+00:00 | Confiança do consumidor brasileiro começa ano em queda para 
[200] json-ld | 2022-01-25T20:52:00+00:00 | OCDE abre processo para a ad